
# Notebook 3 — Integración Multimodal Estructural

Este notebook integra:

- OCR estructurado (`08_ocr_json`)
- OCR TXT (`09_ocr_txt`)
- Florence-2 contextual (`12_image2text`)
- Tablas markdown TATR (`22_tables_markdown`)

Objetivo:

Construir un documento multimodal estructurado listo para:

- corrección contextual con LLM,
- reorganización documental,
- síntesis final.


In [1]:

import os
import re
import json

from pathlib import Path
from typing import Optional
from collections import defaultdict

import pandas as pd



# Configuración de rutas


In [2]:

# ==========================================
# PROJECT ROOT
# ==========================================

def find_project_root(start: Optional[Path] = None) -> Path:

    start = (start or Path.cwd()).resolve()

    for base in [start, *start.parents]:

        markers = [
            ("data", "notebooks"),
            ("src", "data"),
            ("pyproject.toml",),
            ("requirements.txt",),
        ]

        if any(all((base / m).exists() for m in group) for group in markers):
            return base

    return start


PROJECT_ROOT = find_project_root()

print("PROJECT_ROOT:", PROJECT_ROOT)

# ==========================================
# RUN CONFIG
# ==========================================

RUN_NAME = "paper_run_001"

RUN_DIR = PROJECT_ROOT / "data" / "outputs" / RUN_NAME

print("RUN_DIR:", RUN_DIR)

# ==========================================
# INPUT DIRECTORIES
# ==========================================

INPUT_DIRS = {

    # OCR estructurado
    "ocr_json":
        RUN_DIR / "08_ocr_json",

    # OCR texto plano
    "ocr_txt":
        RUN_DIR / "09_ocr_txt",

    # Florence-2
    "image2text":
        RUN_DIR / "12_image2text",

    # Tablas markdown TATR
    "tables_markdown":
        RUN_DIR / "22_tables_markdown",
}

# ==========================================
# OUTPUT DIRECTORIES
# ==========================================

OUTPUT_DIRS = {

    "root":
        RUN_DIR / "30_multimodal",

    "json":
        RUN_DIR / "30_multimodal" / "json",

    "markdown":
        RUN_DIR / "30_multimodal" / "markdown",

    "debug":
        RUN_DIR / "30_multimodal" / "debug",
}

# ==========================================
# CREATE OUTPUT DIRECTORIES
# ==========================================

for path in OUTPUT_DIRS.values():
    path.mkdir(parents=True, exist_ok=True)

# ==========================================
# IMPORTANT FILES
# ==========================================

CAPTIONS_JSON_PATH = (
    INPUT_DIRS["image2text"]
    / "contextual_image2text_results.json"
)

CAPTIONS_CSV_PATH = (
    INPUT_DIRS["image2text"]
    / "contextual_image2text_results.csv"
)

MULTIMODAL_JSON_PATH = (
    OUTPUT_DIRS["json"]
    / "document_multimodal.json"
)

MULTIMODAL_MD_PATH = (
    OUTPUT_DIRS["markdown"]
    / "document_multimodal.md"
)

MULTIMODAL_DEBUG_PATH = (
    OUTPUT_DIRS["debug"]
    / "multimodal_debug.json"
)

print("\n========== INPUTS ==========")

for k, v in INPUT_DIRS.items():
    print(f"{k}: {v}")

print("\n========== OUTPUTS ==========")

for k, v in OUTPUT_DIRS.items():
    print(f"{k}: {v}")


PROJECT_ROOT: C:\Users\user\proyectos\vision-docs
RUN_DIR: C:\Users\user\proyectos\vision-docs\data\outputs\paper_run_001

========== INPUTS ==========
ocr_json: C:\Users\user\proyectos\vision-docs\data\outputs\paper_run_001\08_ocr_json
ocr_txt: C:\Users\user\proyectos\vision-docs\data\outputs\paper_run_001\09_ocr_txt
image2text: C:\Users\user\proyectos\vision-docs\data\outputs\paper_run_001\12_image2text
tables_markdown: C:\Users\user\proyectos\vision-docs\data\outputs\paper_run_001\22_tables_markdown

========== OUTPUTS ==========
root: C:\Users\user\proyectos\vision-docs\data\outputs\paper_run_001\30_multimodal
json: C:\Users\user\proyectos\vision-docs\data\outputs\paper_run_001\30_multimodal\json
markdown: C:\Users\user\proyectos\vision-docs\data\outputs\paper_run_001\30_multimodal\markdown
debug: C:\Users\user\proyectos\vision-docs\data\outputs\paper_run_001\30_multimodal\debug



# Loaders


In [3]:

def load_json_folder(folder_path):

    data = []

    json_files = sorted(folder_path.glob("*.json"))

    print(f"JSON encontrados: {len(json_files)}")

    for file in json_files:

        try:

            with open(file, "r", encoding="utf-8") as f:

                content = json.load(f)

                if isinstance(content, list):
                    data.extend(content)
                else:
                    data.append(content)

        except Exception as e:

            print(f"[ERROR] {file.name}: {e}")

    return data


In [4]:

def load_markdown_tables(folder_path):

    tables = []

    md_files = sorted(folder_path.glob("*.md"))

    print(f"Markdown tables encontradas: {len(md_files)}")

    for idx, md_file in enumerate(md_files):

        with open(md_file, "r", encoding="utf-8") as f:

            markdown_content = f.read()

        filename = md_file.stem

        page_match = re.search(r"page[_-](\d+)", filename)

        if page_match:
            page_num = int(page_match.group(1))
        else:
            page_num = -1

        tables.append({
            "page_num": page_num,
            "bbox": [0, 0, 0, 0],
            "markdown": markdown_content,
            "source_file": md_file.name
        })

    return tables



# Parsers multimodales


In [5]:

def parse_ocr_block(block):

    full_text = "\n".join(
        line["text"]
        for line in block.get("ocr_lines", [])
    )

    return {
        "page": block["page_num"],
        "type": "text",
        "bbox": block["region"]["bbox"],
        "content": full_text,
        "source": "ocr_json"
    }


In [6]:

def parse_caption_block(block):

    return {
        "page": block["page_num"],
        "type": "figure",
        "bbox": block["bbox"],
        "content": block["contextual_description"],
        "source": "florence2"
    }


In [7]:

def parse_table_block(block):

    return {
        "page": block["page_num"],
        "type": "table",
        "bbox": block["bbox"],
        "content": block["markdown"],
        "source": "table_transformer",
        "source_file": block["source_file"]
    }



# Cargar OCR


In [8]:

ocr_raw = load_json_folder(
    INPUT_DIRS["ocr_json"]
)

ocr_blocks = [
    parse_ocr_block(block)
    for block in ocr_raw
]

print("OCR blocks:", len(ocr_blocks))


JSON encontrados: 30
OCR blocks: 30



# Cargar Florence-2


In [9]:

with open(
    CAPTIONS_JSON_PATH,
    "r",
    encoding="utf-8"
) as f:

    captions_raw = json.load(f)

caption_blocks = [
    parse_caption_block(block)
    for block in captions_raw
]

print("Caption blocks:", len(caption_blocks))


Caption blocks: 1



# Cargar tablas markdown


In [10]:

tables_raw = load_markdown_tables(
    INPUT_DIRS["tables_markdown"]
)

table_blocks = [
    parse_table_block(block)
    for block in tables_raw
]

print("Table blocks:", len(table_blocks))


Markdown tables encontradas: 1
Table blocks: 1



# Integración multimodal


In [11]:

all_blocks = (
    ocr_blocks +
    caption_blocks +
    table_blocks
)

print("Total multimodal blocks:", len(all_blocks))


Total multimodal blocks: 32



# Ordenamiento espacial


In [12]:

def sort_blocks(blocks):

    return sorted(
        blocks,
        key=lambda b: (
            b["page"],
            b["bbox"][1],
            b["bbox"][0]
        )
    )


In [13]:

all_blocks = sort_blocks(all_blocks)



# Preview


In [14]:

preview_df = pd.DataFrame(all_blocks)

preview_df.head(20)


,page,type,bbox,content,source,source_file
0,-1,table,"[0, 0, 0, 0]",| 0 | 1 | 2 ...,table_transformer,table_0013.md
1,1,text,"[648, 633, 1315, 665]",,ocr_json,NaN
2,1,text,"[430, 713, 1473, 744]",,ocr_json,NaN
3,1,text,"[424, 934, 1479, 1330]",El objetivo de este estudio fue aislar hongos ...,ocr_json,NaN
4,1,text,"[427, 1373, 1478, 1438]",Palabras claves: dermatofitosis: virus inmunod...,ocr_json,NaN
5,1,text,"[421, 1664, 1482, 1807]",The aim of this study was to isolate dermatoph...,ocr_json,NaN
6,1,text,"[319, 2225, 926, 2291]",Recibido: 1 de octubre de 2018\nAceptado para ...,ocr_json,NaN
7,2,text,"[424, 278, 1483, 493]",Santiago de Chile. Hair and skin scale samples...,ocr_json,NaN
8,2,text,"[432, 536, 1478, 600]",Key words: dermatophytosis; feline immunodefic...,ocr_json,NaN
9,2,text,"[992, 789, 1585, 1543]",Diversos autores han reportado la pre\nsencia ...,ocr_json,NaN



# Construcción del documento estructurado


In [15]:

pages = defaultdict(list)

for block in all_blocks:
    pages[block["page"]].append(block)


In [16]:

document = {
    "run_name": RUN_NAME,
    "pages": []
}

for page_num in sorted(pages.keys()):

    page_data = {
        "page": page_num,
        "blocks": pages[page_num]
    }

    document["pages"].append(page_data)

print("Pages:", len(document["pages"]))


Pages: 6



# Export JSON multimodal


In [17]:

with open(
    MULTIMODAL_JSON_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        document,
        f,
        indent=4,
        ensure_ascii=False
    )

print("JSON exportado:", MULTIMODAL_JSON_PATH)


JSON exportado: C:\Users\user\proyectos\vision-docs\data\outputs\paper_run_001\30_multimodal\json\document_multimodal.json



# Conversión a markdown enriquecido


In [18]:

def block_to_markdown(block):

    if block["type"] == "text":

        return block["content"]

    elif block["type"] == "table":

        return f"\n\n## Tabla\n\n{block['content']}\n"

    elif block["type"] == "figure":

        return f"\n\n## Figura\n\n{block['content']}\n"

    return ""


In [19]:

markdown_document = ""

for page in document["pages"]:

    markdown_document += f"\n\n# Página {page['page']}\n\n"

    for block in page["blocks"]:

        markdown_document += block_to_markdown(block)
        markdown_document += "\n"



# Export markdown multimodal


In [20]:

with open(
    MULTIMODAL_MD_PATH,
    "w",
    encoding="utf-8"
) as f:

    f.write(markdown_document)

print("Markdown exportado:", MULTIMODAL_MD_PATH)


Markdown exportado: C:\Users\user\proyectos\vision-docs\data\outputs\paper_run_001\30_multimodal\markdown\document_multimodal.md



# Preview markdown


In [21]:

print(markdown_document[:5000])




# Página -1



## Tabla

| 0          | 1              | 2              | 3         |
|:-----------|:---------------|:---------------|:----------|
| Retrovirus | Negativos a    | Positivos a    | Total     |
|            | dermatofitosis | dermatofitosis | % (n)     |
|            | % (n)          | % (n)          |           |
| VLeF -VIF  | nan            | 5.7 (2)        | 5.7 (2)   |
| VLeF       | 20.0 (7)       | 48.6 (17)      | 68.6 (24) |
| VIF        | 11.4 (4)       | 14.3 (5)       | 25.7 (9)  |
| Total      | 31.4 (11)      | 68.6 (24)      | 100 (35)  |



# Página 1



El objetivo de este estudio fue aislar hongos dermatofitos desde lesiones dermica
presentes en gatos domésticos (Felis catus) positivos a los retrovirus virus de la
inmunodeficiencia felina (VIF) y virus de la leucemia felina (VLeF). Fueron estudiados 35
felinos: 9 positivos a VIF, 24 a VLeF y 2 a ambos virus, atendidos en la clinica veterinaria
de la Universidad Santo Tomas de Santiago de Chile. Las mue


# Variable final para LLM

Esta variable será usada por:

`notebook_4_llm_correction_summarization.ipynb`


In [22]:

llm_input = markdown_document
